# Lab 05 — MCMC, Systematic Covariance and Nuisance Degeneracy

**Format:** 4-hour hands-on practical + graded extension  
**Status:** **Graded Lab 2 (5%)**  
**Real data:** Pantheon+ / SH0ES  
**Covariances:** published STATONLY and STAT+SYS matrices

> ## Scientific question
> **Can an MCMC sampler reproduce a posterior we already understand, and how do systematic covariances change the scientific uncertainty?**

This lab uses MCMC as an **algorithmic tool**, not as a badge of sophistication.

Because the current posterior has only two parameters, a grid remains the benchmark. This gives us something unusually valuable: we can verify whether the MCMC sampler is telling the truth.

## 0. Five-line statement

**Estimand:**  
**Observed data:**  
**Generative assumptions:**  
**Uncertainty / selection:**  
**Validation / falsification test:**

## 1. Setup

If needed in Colab:

```python
!pip -q install emcee corner
```

In [ ]:
from pathlib import Path
import io, sys, time, requests, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.linalg import cho_factor, cho_solve
from scipy.interpolate import RegularGridInterpolator
from scipy.special import logsumexp
from astropy.cosmology import FlatLambdaCDM

SEED = 2026
rng = np.random.default_rng(SEED)

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

try:
    import emcee
    import corner
except ImportError:
    raise ImportError("Install emcee and corner first: pip install emcee corner")

In [ ]:
BASE = (
    "https://raw.githubusercontent.com/PantheonPlusSH0ES/"
    "DataRelease/main/Pantheon%2B_Data/4_DISTANCES_AND_COVAR/"
)
URL_DATA = BASE + "Pantheon%2BSH0ES.dat"
URL_STAT = BASE + "Pantheon%2BSH0ES_STATONLY.cov"
URL_FULL = BASE + "Pantheon%2BSH0ES_STAT%2BSYS.cov"

CACHE = Path("data_cache/pantheon_plus")
CACHE.mkdir(parents=True, exist_ok=True)

def download_if_needed(url, path, timeout=180):
    path = Path(path)
    if not path.exists():
        print("Downloading", path.name, "...")
        r = requests.get(url, timeout=timeout)
        r.raise_for_status()
        path.write_bytes(r.content)
    return path

def load_pantheon_table():
    p = download_if_needed(URL_DATA, CACHE/"Pantheon+SH0ES.dat")
    return pd.read_csv(p, sep=r"\s+")

def load_covariance(kind="STATONLY"):
    if kind.upper() == "STATONLY":
        url = URL_STAT
        path = CACHE/"Pantheon+SH0ES_STATONLY.cov"
    elif kind.upper() in {"FULL","STAT+SYS","STATSYS"}:
        url = URL_FULL
        path = CACHE/"Pantheon+SH0ES_STAT+SYS.cov"
    else:
        raise ValueError("kind must be STATONLY or FULL")

    p = download_if_needed(url, path)
    flat = np.loadtxt(p)
    n = int(flat[0])
    values = flat[1:]
    if values.size != n*n:
        raise ValueError(f"Expected {n*n} covariance entries, found {values.size}")
    return values.reshape(n, n)

sn = load_pantheon_table()
print("Pantheon+ rows:", len(sn))

In [ ]:
def teaching_mask(sn):
    """Real-data teaching sample: non-Cepheid calibrators with zHD > 0.01."""
    mask = (
        np.isfinite(sn["zHD"].to_numpy(float)) &
        np.isfinite(sn["m_b_corr"].to_numpy(float)) &
        (sn["zHD"].to_numpy(float) > 0.01) &
        (sn["IS_CALIBRATOR"].to_numpy(int) == 0)
    )
    return mask

def subset_data_cov(sn, cov, mask):
    idx = np.flatnonzero(mask)
    d = sn.iloc[idx].reset_index(drop=True).copy()
    c = cov[np.ix_(idx, idx)]
    return d, c, idx

def mu_flat_lcdm(z, omega_m, H0=70.0):
    cosmo = FlatLambdaCDM(H0=H0, Om0=omega_m, Tcmb0=2.725)
    return cosmo.distmod(np.asarray(z, float)).value

def coefficient_grid(data, cov, omega_grid, H0=70.0):
    """
    For model m_b_corr = mu(z; Omega_m,H0) + M_B,
    compute coefficients so that
    chi2(Omega_m, M_B) = a - 2 M_B b + M_B^2 d.
    """
    z = data["zHD"].to_numpy(float)
    m = data["m_b_corr"].to_numpy(float)
    one = np.ones(len(data))

    cf = cho_factor(cov, lower=True, check_finite=False)
    cinv_one = cho_solve(cf, one, check_finite=False)
    dcoef = one @ cinv_one

    a = np.empty(len(omega_grid))
    b = np.empty(len(omega_grid))

    for k, om in enumerate(omega_grid):
        y = m - mu_flat_lcdm(z, om, H0=H0)
        cinv_y = cho_solve(cf, y, check_finite=False)
        a[k] = y @ cinv_y
        b[k] = one @ cinv_y

    return a, b, float(dcoef)

def profile_mb(a, b, d):
    mb_hat = b/d
    chi2_prof = a - b*b/d
    return mb_hat, chi2_prof

def chi2_surface(a, b, d, mb_grid):
    mb = np.asarray(mb_grid, float)[None, :]
    return a[:,None] - 2*b[:,None]*mb + d*mb*mb

# Part I — Statistical versus statistical+systematic covariance

## 2. Load both published covariance matrices

The Pantheon+ data release distinguishes:

\[
C_{\rm stat}
\]

and

\[
C_{\rm stat+sys}.
\]

We use exactly the same real SN rows for both likelihoods.

In [ ]:
cov_stat_all = load_covariance("STATONLY")
cov_full_all = load_covariance("FULL")

mask = teaching_mask(sn)
data, Cstat, idx = subset_data_cov(sn, cov_stat_all, mask)
_, Cfull, idx2 = subset_data_cov(sn, cov_full_all, mask)

assert np.array_equal(idx, idx2)

print("Teaching sample:", len(data))
print("STAT covariance:", Cstat.shape)
print("FULL covariance:", Cfull.shape)

### Exercise 1 — covariance structure

For each matrix report:

- median diagonal standard deviation;
- median absolute off-diagonal covariance;
- maximum absolute correlation coefficient off diagonal.

Then plot a small \(100\times100\) correlation-matrix block for each.

**Question:** What does “systematic uncertainty” mean here statistically? It is not merely “add a larger error bar to every SN.”

## 3. Build posterior grids for both covariance choices

Use identical parameter grids and priors:

\[
0.10\le\Omega_m\le0.60,
\qquad
-20.2\le M_B\le-18.2.
\]

The only change is the covariance model.

In [ ]:
OMEGA_GRID = np.linspace(0.10,0.60,101)
MB_GRID = np.linspace(-20.2,-18.2,161)
H0_CONVENTION = 70.0

a_s,b_s,d_s = coefficient_grid(data,Cstat,OMEGA_GRID,H0=H0_CONVENTION)
a_f,b_f,d_f = coefficient_grid(data,Cfull,OMEGA_GRID,H0=H0_CONVENTION)

chi2_s = chi2_surface(a_s,b_s,d_s,MB_GRID)
chi2_f = chi2_surface(a_f,b_f,d_f,MB_GRID)

def normalized_posterior_from_chi2(chi2):
    lp = -0.5*chi2
    lp -= logsumexp(lp)
    p = np.exp(lp)
    return p/p.sum()

P_stat = normalized_posterior_from_chi2(chi2_s)
P_full = normalized_posterior_from_chi2(chi2_f)

p_om_stat = P_stat.sum(axis=1); p_om_stat /= p_om_stat.sum()
p_om_full = P_full.sum(axis=1); p_om_full /= p_om_full.sum()

fig, ax = plt.subplots(figsize=(7,4))
ax.plot(OMEGA_GRID,p_om_stat,label="STATONLY")
ax.plot(OMEGA_GRID,p_om_full,label="STAT+SYS")
ax.set_xlabel(r"$\Omega_m$")
ax.set_ylabel("Marginal posterior")
ax.legend()
plt.show()

### Exercise 2 — quantify the systematic-covariance effect

Create:

| covariance | median \(\Omega_m\) | 16th | 84th | 68% width |
|---|---:|---:|---:|---:|

Do the same for \(M_B\).

Then answer:

> Did systematics mainly shift the posterior, broaden it, change its correlation structure, or some combination?

# Part II — MCMC as a numerical algorithm

## 4. Why sample an interpolated posterior?

Evaluating the full covariance likelihood at every MCMC step would be unnecessarily expensive for a classroom exercise.

We have already evaluated the **real full-covariance posterior** on a dense grid. We now create a smooth interpolator of its log posterior and ask MCMC to recover that target.

This separates two questions:

1. **Scientific likelihood construction:** already done using the real covariance.
2. **Posterior exploration algorithm:** tested here with MCMC.

In a higher-dimensional research problem, the likelihood would normally be evaluated directly or via a validated surrogate/compression.

In [ ]:
logP_full_grid = np.log(P_full + 1e-300)

interp = RegularGridInterpolator(
    (OMEGA_GRID, MB_GRID),
    logP_full_grid,
    bounds_error=False,
    fill_value=-np.inf
)

def log_prob(theta):
    om, mb = theta
    if not (OMEGA_GRID[0] <= om <= OMEGA_GRID[-1]):
        return -np.inf
    if not (MB_GRID[0] <= mb <= MB_GRID[-1]):
        return -np.inf
    return float(interp([[om,mb]])[0])

## 5. Initialize and run `emcee`

Use multiple walkers.

Do not initialize every walker at exactly the same point.

In [ ]:
nwalkers = 32
ndim = 2
nsteps = 2500

# Start near the full-grid posterior maximum.
imax = np.unravel_index(np.argmax(P_full), P_full.shape)
start_center = np.array([OMEGA_GRID[imax[0]], MB_GRID[imax[1]]])

pos = start_center + rng.normal(scale=[0.01,0.01], size=(nwalkers,ndim))

sampler = emcee.EnsembleSampler(nwalkers, ndim, log_prob)
sampler.run_mcmc(pos, nsteps, progress=True)

print("Mean acceptance fraction:", np.mean(sampler.acceptance_fraction))

# Part III — Diagnose before quoting

## 6. Trace plots

A corner plot is **not** a convergence diagnostic.

Inspect each walker as a function of step.

In [ ]:
chain = sampler.get_chain()

fig, axes = plt.subplots(2,1, figsize=(9,6), sharex=True)
labels = [r"$\Omega_m$", r"$M_B$"]

for j, ax in enumerate(axes):
    ax.plot(chain[:,:,j], alpha=0.25)
    ax.set_ylabel(labels[j])
axes[-1].set_xlabel("MCMC step")
plt.tight_layout()
plt.show()

### Exercise 3 — burn-in and autocorrelation

1. Estimate integrated autocorrelation time with `sampler.get_autocorr_time(tol=0)`.
2. Choose burn-in based on the traces and autocorrelation estimate.
3. Choose thinning only if you can justify it. **Thinning is not automatically required.**
4. Compute an approximate effective sample size.

If the chain is too short relative to the autocorrelation time, extend it.

In [ ]:
try:
    tau = sampler.get_autocorr_time(tol=0)
    print("Autocorrelation time:", tau)
except Exception as e:
    print("Autocorrelation estimate warning:", e)

# TODO: choose burn-in after inspecting traces.
burnin = 500
flat = sampler.get_chain(discard=burnin, flat=True)
print("Retained samples:", len(flat))

## 7. Corner plot and grid validation

Overlay or compare MCMC summaries with the **known grid marginal posterior**.

If MCMC disagrees with the grid beyond Monte-Carlo noise, the sampler/configuration has failed.

In [ ]:
fig = corner.corner(
    flat,
    labels=[r"$\Omega_m$",r"$M_B$"],
    truths=[OMEGA_GRID[imax[0]],MB_GRID[imax[1]]]
)
plt.show()

def q16_50_84(x):
    return np.percentile(x,[16,50,84])

print("MCMC Omega_m:", q16_50_84(flat[:,0]))
print("MCMC M_B:", q16_50_84(flat[:,1]))

### Exercise 4 — sampler validation

Compare:

- grid posterior median and 68% interval;
- MCMC median and 68% interval.

Create a table and calculate the absolute differences.

Then write:

> “MCMC is validated here because …”

or

> “MCMC is not yet validated because …”

# 8. In-class checkpoint

Show:

1. STATONLY versus STAT+SYS covariance diagnostic;
2. marginal posterior comparison;
3. MCMC trace plots;
4. acceptance fraction;
5. autocorrelation estimate;
6. corner plot;
7. MCMC-versus-grid comparison.

You should not leave class with only a corner plot.

# Extended Analysis — Graded Lab 2

## E1. Systematic covariance table

Produce the full comparison for \(\Omega_m\) and \(M_B\):

| parameter | STAT median | STAT 68% | STAT+SYS median | STAT+SYS 68% | shift |
|---|---:|---:|---:|---:|---:|

Interpret both the shift and the interval-width change.

## E2. \(H_0\)–\(M_B\) zero-point degeneracy

Repeat the full-covariance coefficient grid with

\[
H_0=73.
\]

Compare with \(H_0=70\).

Report:

\[
\Delta \widehat M_B
=
\widehat M_B(H_0=73)-\widehat M_B(H_0=70).
\]

Compare the numerical shift with

\[
5\log_{10}\left(\frac{73}{70}\right).
\]

What happens to \(\Omega_m\)?

Explain why this is an identifiability issue, not evidence for one \(H_0\) value.

## E3. MCMC reproducibility

Repeat the MCMC with a different random seed.

Compare:

- posterior medians;
- 68% intervals;
- acceptance fraction;
- autocorrelation estimate.

The chains need not match point by point.

## E4. Deliberate bad run

Run one intentionally inadequate chain, for example only 100–200 steps.

Compare its traces and posterior summary with the well-run chain.

Explain why a plausible corner plot can still be misleading.

## E5. Scientific conclusion — 300–500 words

Answer:

1. What did adding systematic covariance do?
2. What is the nuisance calibration degeneracy?
3. How did you establish MCMC convergence/reliability?
4. What is your principal numerical cosmology result **conditional on this teaching model**?
5. Why is it not an independent measurement of \(H_0\)?
6. What would be irresponsible to claim from this lab?

# 9. Submission checklist

The same notebook is both the lab and the graded extension.

Required:

- covariance comparison;
- posterior comparison;
- trace plots;
- autocorrelation/acceptance diagnostics;
- MCMC-versus-grid validation;
- \(H_0\)-convention test;
- second-seed reproducibility check;
- deliberately bad-chain comparison;
- 300–500 word Scientific Conclusion.

**Rubric:** formulation 20%, statistical correctness 25%, uncertainty 20%, diagnostics 15%, reproducibility 10%, interpretation 10%.

# 10. Bridge to Lab 06

You now have a statistically coherent cosmological likelihood.

Next question:

> **How stable is that inference when we change the real observational sample?**

Lab 06 therefore moves from parameter uncertainty **within a fixed dataset** to robustness under defensible real-data sample definitions.